In [0]:
pip install findspark

In [0]:
import findspark, pyspark
from pyspark.sql import SparkSession
findspark.init()
spark = SparkSession.builder.appName('kmeans').getOrCreate()

In [0]:
abs_path = "/Volumes/workspace/ml_with_spark/raw_data/iris.csv"

iris = spark.read.csv(abs_path, header=True, inferSchema=True)
iris.show(5)

In [0]:
from pyspark.ml.clustering import KMeans

In [0]:
from pyspark.ml.feature import VectorAssembler

asb = VectorAssembler(inputCols=iris.columns[:-1], outputCol='features')
iris_ass = asb.transform(iris)

iris_ass.show(5)

In [0]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol='class', outputCol='class_trans')
iris_ass = indexer.fit(iris_ass).transform(iris_ass)
iris_ass.show(5)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

iris_ass = iris_ass.withColumn("class_trans", col("class_trans").cast(IntegerType()))
iris_ass.show(5)

In [0]:
km = KMeans(
    predictionCol="grupo",
    maxIter=100,
    k=3,
    featuresCol="features"
)

modelo = km.fit(iris_ass)

In [0]:
grupos = modelo.transform(iris_ass)

grupos.show(5)

In [0]:
classe = grupos.select("class_trans").collect()

agrupado = grupos.select("grupo").collect()

In [0]:
from sklearn.metrics import confusion_matrix, accuracy_score

cm = confusion_matrix(classe, agrupado)

print(cm)

In [0]:
acuracia = (cm[0,1] + cm[1,0] + cm[2,2]) / 150
acuracia